2

In [5]:
import os
import sys

project_path = os.path.abspath("..")

if project_path not in sys.path:
    sys.path.append(project_path)

print("Project path:", project_path)

Project path: c:\Users\Akshaya R\Desktop\olist-data-engineering-pipeline


In [7]:
import pandas as pd
from pathlib import Path

from src.reader import read_all_csv

In [8]:
dataset_path = Path("../dataset")

data = read_all_csv(dataset_path)

print("Total CSV files loaded:", len(data))

File loaded: olist_customers_dataset.csv
Total rows: 99441
Total columns: 5
---------------------------
File loaded: olist_geolocation_dataset.csv
Total rows: 1000163
Total columns: 5
---------------------------
File loaded: olist_orders_dataset.csv
Total rows: 99441
Total columns: 8
---------------------------
File loaded: olist_order_items_dataset.csv
Total rows: 112650
Total columns: 7
---------------------------
File loaded: olist_order_payments_dataset.csv
Total rows: 103886
Total columns: 5
---------------------------
File loaded: olist_order_reviews_dataset.csv
Total rows: 99224
Total columns: 7
---------------------------
File loaded: olist_products_dataset.csv
Total rows: 32951
Total columns: 9
---------------------------
File loaded: olist_sellers_dataset.csv
Total rows: 3095
Total columns: 4
---------------------------
File loaded: product_category_name_translation.csv
Total rows: 71
Total columns: 2
---------------------------
Total CSV files loaded: 9


3

In [9]:
import yaml

with open("../config/config.yaml", "r") as file:
    config = yaml.safe_load(file)

print(config)

{'column_mapping': {'olist_orders_dataset': {'order_purchase_timestamp': 'purchase_timestamp', 'order_approved_at': 'approved_timestamp', 'order_delivered_carrier_date': 'carrier_delivery_timestamp', 'order_delivered_customer_date': 'customer_delivery_timestamp', 'order_estimated_delivery_date': 'estimated_delivery_timestamp'}, 'olist_customers_dataset': {'customer_zip_code_prefix': 'zip_code_prefix', 'customer_city': 'city', 'customer_state': 'state'}, 'olist_products_dataset': {'product_category_name': 'category_name', 'product_name_lenght': 'product_name_length', 'product_description_lenght': 'product_description_length'}}}


In [10]:
print("Available datasets:")

for dataset in config["column_mapping"]:
    print(dataset)

Available datasets:
olist_orders_dataset
olist_customers_dataset
olist_products_dataset


In [11]:
print(config["column_mapping"]["olist_orders_dataset"])

{'order_purchase_timestamp': 'purchase_timestamp', 'order_approved_at': 'approved_timestamp', 'order_delivered_carrier_date': 'carrier_delivery_timestamp', 'order_delivered_customer_date': 'customer_delivery_timestamp', 'order_estimated_delivery_date': 'estimated_delivery_timestamp'}


In [12]:
from src.transformer import apply_column_mapping

transformed_data = apply_column_mapping(data, config)

Mapping completed: olist_customers_dataset
Mapping completed: olist_geolocation_dataset
Mapping completed: olist_orders_dataset
Mapping completed: olist_order_items_dataset
Mapping completed: olist_order_payments_dataset
Mapping completed: olist_order_reviews_dataset
Mapping completed: olist_products_dataset
Mapping completed: olist_sellers_dataset
Mapping completed: product_category_name_translation


In [13]:
print(transformed_data["olist_orders_dataset"].columns.tolist())

['order_id', 'customer_id', 'order_status', 'purchase_timestamp', 'approved_timestamp', 'carrier_delivery_timestamp', 'customer_delivery_timestamp', 'estimated_delivery_timestamp']


In [14]:
print(transformed_data["olist_customers_dataset"].columns.tolist())

['customer_id', 'customer_unique_id', 'zip_code_prefix', 'city', 'state']


4

In [17]:
import yaml

with open("../config/config.yaml", "r") as file:
    config = yaml.safe_load(file)

print(config.keys())

dict_keys(['column_mapping', 'validation'])


In [18]:
from src.validator import validate_data

valid_data, bad_records = validate_data(
    transformed_data,
    config
)

olist_customers_dataset: Valid = 99441, Invalid = 0
olist_orders_dataset: Valid = 99441, Invalid = 0
olist_products_dataset: Valid = 32951, Invalid = 0


In [19]:
print("Total bad records:", len(bad_records))

Total bad records: 0


In [20]:
from pathlib import Path

output_path = Path("../output")
output_path.mkdir(parents=True, exist_ok=True)

bad_records.to_csv(
    output_path / "bad_records.csv",
    index=False
)

print("Bad records file created successfully")
print("Total bad records:", len(bad_records))

Bad records file created successfully
Total bad records: 0


In [21]:
bad_records_file = output_path / "bad_records.csv"

print("File exists:", bad_records_file.exists())
print("File location:", bad_records_file)

File exists: True
File location: ..\output\bad_records.csv


In [ ]:
5

In [22]:
from src.exceptions import (
    PipelineError,
    DatasetNotFoundError,
    MissingRequiredColumnsError,
    DataValidationError
)

try:
    raise DatasetNotFoundError("Olist dataset file not found")

except DatasetNotFoundError as error:
    print("Dataset Error:", error)

print("Pipeline execution continues.")

Dataset Error: Olist dataset file not found
Pipeline execution continues.


In [24]:
import importlib

import src.exceptions
import src.validator

importlib.reload(src.exceptions)
importlib.reload(src.validator)

from src.exceptions import MissingRequiredColumnsError
from src.validator import validate_data

In [25]:
try:
    test_data = {
        "olist_orders_dataset": transformed_data[
            "olist_orders_dataset"
        ].drop(columns=["order_id"])
    }

    validate_data(test_data, config)

except MissingRequiredColumnsError as error:
    print("Validation Error:", error)

print("Notebook execution continues.")

Validation Error: Missing required columns in olist_orders_dataset: ['order_id']
Notebook execution continues.


In [26]:
from src.pipeline import run_pipeline

run_pipeline()

Pipeline execution started
Pipeline execution completed successfully


6

In [27]:
from src.pipeline import run_pipeline

run_pipeline()

Pipeline execution started
Pipeline execution completed successfully


In [29]:
from pathlib import Path

print("Notebook location:", Path.cwd())
print("Project folder contents:")

for item in Path.cwd().iterdir():
    print(item.name)

Notebook location: c:\Users\Akshaya R\Desktop\olist-data-engineering-pipeline\notebooks
Project folder contents:
day15.ipynb


In [31]:
project_path = Path.cwd()

for log_file in project_path.rglob("pipeline.log"):
    print("Log file found:", log_file.resolve())

In [32]:
import sys
from pathlib import Path
import src.pipeline

print("Python interpreter:", sys.executable)
print("Pipeline module:", src.pipeline.__file__)

project_path = Path.cwd().parent
log_folder = project_path / "logs"
log_file = log_folder / "pipeline.log"

print("Project folder:", project_path)
print("Logs folder exists:", log_folder.exists())
print("Log file exists:", log_file.exists())

Python interpreter: c:\Python315\python.exe
Pipeline module: c:\Users\Akshaya R\Desktop\olist-data-engineering-pipeline\src\pipeline.py
Project folder: c:\Users\Akshaya R\Desktop\olist-data-engineering-pipeline
Logs folder exists: True
Log file exists: False


In [33]:
import importlib
import src.pipeline

# Reload the latest pipeline.py code
importlib.reload(src.pipeline)

# Run the pipeline
src.pipeline.run_pipeline()

In [34]:
from pathlib import Path

log_file = Path("../logs/pipeline.log")

print("Log file exists:", log_file.exists())

if log_file.exists():
    print(log_file.read_text())
    

Log file exists: True
2026-11-24 21:15:02,544 - INFO - Pipeline execution started
2026-11-24 21:15:02,544 - INFO - Pipeline execution completed successfully



7

In [36]:
import importlib
import src.pipeline

importlib.reload(src.pipeline)

src.pipeline.run_pipeline()

Processing new file: olist_customers_dataset.csv
Processing new file: olist_geolocation_dataset.csv
Processing new file: olist_orders_dataset.csv
Processing new file: olist_order_items_dataset.csv
Processing new file: olist_order_payments_dataset.csv
Processing new file: olist_order_reviews_dataset.csv
Processing new file: olist_products_dataset.csv
Processing new file: olist_sellers_dataset.csv
Processing new file: product_category_name_translation.csv


In [37]:
{
    "processed_files": [
        "olist_customers_dataset.csv",
        "olist_orders_dataset.csv",
        "olist_geolocation_dataset.csv"
    ]
}

{'processed_files': ['olist_customers_dataset.csv',
  'olist_orders_dataset.csv',
  'olist_geolocation_dataset.csv']}

In [38]:
manifest = src.pipeline.load_manifest()

print("Total processed files:", len(manifest["processed_files"]))

for file in manifest["processed_files"]:
    print(file)
    

Total processed files: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [ ]:
8

In [44]:
import sys
from pathlib import Path

project_path = Path.cwd().parent

print("Project path:", project_path)
print("Virtual environment:", project_path / "venv")

Project path: c:\Users\Akshaya R\Desktop\olist-data-engineering-pipeline
Virtual environment: c:\Users\Akshaya R\Desktop\olist-data-engineering-pipeline\venv


In [46]:
import subprocess

python_path = project_path / "venv" / "Scripts" / "python.exe"

result = subprocess.run(
    [
        str(python_path),
        "-m",
        "pytest",
        "tests/",
        "-v"
    ],
    cwd=project_path,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode == 0:
    print("All unit tests passed successfully!")
else:
    print(result.stderr)

============================= test session starts =============================
platform win32 -- Python 3.15.2, pytest-9.2.0, pluggy-1.7.0 -- c:\Users\Akshaya R\Desktop\olist-data-engineering-pipeline\venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\Akshaya R\Desktop\olist-data-engineering-pipeline
collecting ... collected 5 items

tests/test_reader.py::test_read_all_csv PASSED                           [ 20%]
tests/test_reader.py::test_read_all_csv_columns PASSED                   [ 40%]
tests/test_transformer.py::test_apply_column_mapping PASSED              [ 60%]
tests/test_validator.py::test_validate_data_success PASSED               [ 80%]
tests/test_validator.py::test_validate_data_missing_columns PASSED       [100%]

============================== 5 passed in 0.42s ==============================

All unit tests passed successfully!


9

In [48]:
import pandas as pd

# Create sample data with an extra column
new_customer_data = {
    "customer_id": ["C001", "C002"],
    "customer_unique_id": ["U001", "U002"],
    "customer_zip_code_prefix": [625001, 625002],
    "customer_city": ["madurai", "chennai"],
    "customer_state": ["TN", "TN"],
    "customer_type": ["Regular", "Premium"]
}

new_df = pd.DataFrame(new_customer_data)

new_data = {
    "olist_customers_dataset": new_df
}

# Apply existing configuration
updated_data = apply_column_mapping(new_data, config)

print("Updated columns:")
print(updated_data["olist_customers_dataset"].columns.tolist())

Mapping completed: olist_customers_dataset
Updated columns:
['customer_id', 'customer_unique_id', 'zip_code_prefix', 'city', 'state', 'customer_type']


In [51]:
import yaml

with open("../config/config.yaml", "r") as file:
    config = yaml.safe_load(file)

print(config["column_mapping"]["olist_customers_dataset"])

{'customer_zip_code_prefix': 'zip_code_prefix', 'customer_city': 'city', 'customer_state': 'state', 'customer_type': 'customer_segment'}


In [52]:
updated_data = apply_column_mapping(new_data, config)

print("Updated columns:")
print(updated_data["olist_customers_dataset"].columns.tolist())

Mapping completed: olist_customers_dataset
Updated columns:
['customer_id', 'customer_unique_id', 'zip_code_prefix', 'city', 'state', 'customer_segment']
